# Ultrasound — BrEaST external evaluation

Held-out evaluation of the nine completed BUS-BRA runs on BrEaST using fixed validation-selected thresholds and patient-level statistical analysis.


In [ ]:
!pip -q install scipy tqdm statsmodels

In [ ]:
FROZEN_RUNS = [{'model_name': 'attention_unet', 'seed': 42, 'parameter_count': 7851773, 'best_epoch': 52, 'selected_threshold': 0.4, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9239987833652122, 'test_patient_dice': 0.9206228228812225, 'external_evaluated': False}, {'model_name': 'attention_unet', 'seed': 123, 'parameter_count': 7851773, 'best_epoch': 66, 'selected_threshold': 0.85, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9261475747556638, 'test_patient_dice': 0.9201143991729405, 'external_evaluated': False}, {'model_name': 'attention_unet', 'seed': 2025, 'parameter_count': 7851773, 'best_epoch': 53, 'selected_threshold': 0.6, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9254554777301479, 'test_patient_dice': 0.9199555804201388, 'external_evaluated': False}, {'model_name': 'swin_tiny_unet', 'seed': 42, 'parameter_count': 38350819, 'best_epoch': 21, 'selected_threshold': 0.4, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9274627136667558, 'test_patient_dice': 0.9218055683732196, 'external_evaluated': False}, {'model_name': 'swin_tiny_unet', 'seed': 123, 'parameter_count': 38350819, 'best_epoch': 31, 'selected_threshold': 0.45, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9258396930702165, 'test_patient_dice': 0.9191932729495612, 'external_evaluated': False}, {'model_name': 'swin_tiny_unet', 'seed': 2025, 'parameter_count': 38350819, 'best_epoch': 28, 'selected_threshold': 0.4, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9268005346304816, 'test_patient_dice': 0.919958730852849, 'external_evaluated': False}, {'model_name': 'unet', 'seed': 42, 'parameter_count': 7763041, 'best_epoch': 72, 'selected_threshold': 0.45, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9252317362894822, 'test_patient_dice': 0.9193047897233306, 'external_evaluated': False}, {'model_name': 'unet', 'seed': 123, 'parameter_count': 7763041, 'best_epoch': 60, 'selected_threshold': 0.75, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9229957152917387, 'test_patient_dice': 0.91964404352043, 'external_evaluated': False}, {'model_name': 'unet', 'seed': 2025, 'parameter_count': 7763041, 'best_epoch': 62, 'selected_threshold': 0.7, 'threshold_selection_split': 'validation', 'threshold_selection_level': 'patient', 'validation_patient_dice': 0.9234096636898326, 'test_patient_dice': 0.9187866401414595, 'external_evaluated': False}]

EXPECTED_MODELS = [
    "unet",
    "attention_unet",
    "swin_tiny_unet",
]
EXPECTED_SEEDS = [42, 123, 2025]

IMAGE_SIZE = 256
BATCH_SIZE_BY_MODEL = {
    "unet": 16,
    "attention_unet": 12,
    "swin_tiny_unet": 6,
}
NUM_WORKERS = 2
BOOTSTRAP_REPLICATES = 10000
BOOTSTRAP_SEED = 20260719

DATA_ROOT_OVERRIDE = ""
CHECKPOINT_ROOT_OVERRIDE = ""
OUTPUT_ROOT_OVERRIDE = ""

CREATE_QUALITATIVE_OVERLAYS = True
N_OVERLAYS_PER_MODEL = 12

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import shutil
import sys
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import binary_erosion, distance_transform_edt
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

IN_KAGGLE = (
    Path("/kaggle/input").exists()
    and Path("/kaggle/working").exists()
)

IN_COLAB = False
if not IN_KAGGLE:
    try:
        from google.colab import drive
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as error:
        print("Google Drive mount warning:", error)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
AMP_ENABLED = DEVICE.type == "cuda"
RUNTIME_NAME = (
    "kaggle"
    if IN_KAGGLE
    else ("colab" if IN_COLAB else "local")
)

print({
    "runtime": RUNTIME_NAME,
    "device": str(DEVICE),
    "torch": torch.__version__,
})
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

random.seed(BOOTSTRAP_SEED)
np.random.seed(BOOTSTRAP_SEED)
torch.manual_seed(BOOTSTRAP_SEED)
torch.cuda.manual_seed_all(BOOTSTRAP_SEED)

In [ ]:
frozen_df = pd.DataFrame(FROZEN_RUNS).sort_values(
    ["model_name", "seed"]
).reset_index(drop=True)

expected_pairs = {
    (model_name, seed)
    for model_name in EXPECTED_MODELS
    for seed in EXPECTED_SEEDS
}
observed_pairs = set(
    zip(
        frozen_df["model_name"],
        frozen_df["seed"].astype(int),
    )
)

if observed_pairs != expected_pairs:
    raise RuntimeError(
        "Frozen run registry is incomplete. "
        f"Missing={expected_pairs-observed_pairs}; "
        f"extra={observed_pairs-expected_pairs}"
    )

if frozen_df["external_evaluated"].any():
    raise RuntimeError(
        "At least one internal run claims that the external cohort "
        "was already evaluated."
    )

if not (
    frozen_df["threshold_selection_split"]
    .eq("validation")
    .all()
):
    raise RuntimeError(
        "All thresholds must come from validation."
    )

if not (
    frozen_df["threshold_selection_level"]
    .eq("patient")
    .all()
):
    raise RuntimeError(
        "All thresholds must be selected at patient level."
    )

print("Frozen threshold registry: PASS")
display(
    frozen_df[
        [
            "model_name",
            "seed",
            "best_epoch",
            "selected_threshold",
            "validation_patient_dice",
            "test_patient_dice",
        ]
    ]
)

In [ ]:
# Full dataset discovery
EXPECTED_EXTERNAL_CROPS = 252

AUTO_EXTRACT_ROOT = Path(
    "/kaggle/working/TRACKC_EXTERNAL_AUTO_EXTRACTED"
    if IN_KAGGLE
    else "/content/TRACKC_EXTERNAL_AUTO_EXTRACTED"
)

def safe_extract_zip(
    zip_path: Path,
    destination: Path,
) -> Path:
    marker = destination / ".extraction_complete"

    if marker.exists():
        return destination

    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        root = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(
                    f"Unsafe ZIP member: {member.filename}"
                )
        archive.extractall(destination)

    marker.write_text(str(zip_path), encoding="utf-8")
    print("Extracted:", zip_path, "->", destination)
    return destination

def search_roots() -> List[Path]:
    roots = []

    if IN_KAGGLE:
        roots.extend([
            Path("/kaggle/input"),
            Path("/kaggle/working"),
        ])

    if IN_COLAB:
        roots.extend([
            Path("/content/drive/MyDrive/TRACKC_ULTRASOUND"),
            Path("/content"),
        ])

    roots.extend([Path.cwd(), Path("/mnt/data")])

    unique = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        key = str(root.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(root)
    return unique

def count_npz(root: Path, limit: Optional[int] = None) -> int:
    count = 0
    try:
        for _ in root.rglob("*.npz"):
            count += 1
            if limit is not None and count >= limit:
                break
    except Exception:
        pass
    return count

def zip_contains_external_dataset(zip_path: Path) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = [
                name.replace("\\", "/")
                for name in archive.namelist()
            ]
        has_manifest = any(
            name.endswith("roi_us_manifest.csv")
            for name in names
        )
        external_npz = sum(
            name.lower().endswith(".npz")
            and "/breast/external/" in name.lower()
            for name in names
        )
        return has_manifest and external_npz >= EXPECTED_EXTERNAL_CROPS
    except Exception:
        return False

def discover_data_root() -> Path:
    if DATA_ROOT_OVERRIDE:
        root = Path(DATA_ROOT_OVERRIDE)
        if not (root / "roi_us_manifest.csv").exists():
            raise FileNotFoundError(
                f"Manifest not found under DATA_ROOT_OVERRIDE: {root}"
            )
        return root

    candidates = []

    for base in search_roots():
        try:
            for manifest_path in base.rglob("roi_us_manifest.csv"):
                root = manifest_path.parent
                external_npz_count = sum(
                    1
                    for path in root.rglob("*.npz")
                    if "breast" in str(path).lower()
                    and "external" in str(path).lower()
                )
                candidates.append(
                    (external_npz_count, root)
                )
        except Exception:
            pass

    valid = [
        item for item in candidates
        if item[0] >= EXPECTED_EXTERNAL_CROPS
    ]

    if not valid:
        for base in search_roots():
            try:
                zip_paths = list(base.rglob("*.zip"))
            except Exception:
                zip_paths = []

            for zip_path in zip_paths:
                if zip_contains_external_dataset(zip_path):
                    destination = (
                        AUTO_EXTRACT_ROOT
                        / re.sub(
                            r"[^A-Za-z0-9_.-]+",
                            "_",
                            zip_path.stem,
                        )
                    )
                    safe_extract_zip(zip_path, destination)

        candidates = []
        for manifest_path in AUTO_EXTRACT_ROOT.rglob(
            "roi_us_manifest.csv"
        ):
            root = manifest_path.parent
            external_npz_count = sum(
                1
                for path in root.rglob("*.npz")
                if "breast" in str(path).lower()
                and "external" in str(path).lower()
            )
            candidates.append(
                (external_npz_count, root)
            )

        valid = [
            item for item in candidates
            if item[0] >= EXPECTED_EXTERNAL_CROPS
        ]

    if not valid:
        raise FileNotFoundError(
            "The full ROI_US_Crops_256_v1 dataset was not found. "
            "Add the full dataset folder or ROI_US_Crops_256_v1_FULL.zip."
        )

    selected_count, selected_root = sorted(
        valid,
        key=lambda item: item[0],
        reverse=True,
    )[0]

    print(
        "Selected dataset root:",
        selected_root,
    )
    print(
        "External NPZ detected:",
        selected_count,
    )
    return selected_root

def discover_output_root() -> Path:
    if OUTPUT_ROOT_OVERRIDE:
        root = Path(OUTPUT_ROOT_OVERRIDE)
    elif IN_KAGGLE:
        root = Path(
            "/kaggle/working/"
            "TRACKC_FINAL_EXTERNAL_EVALUATION"
        )
    elif IN_COLAB:
        root = Path(
            "/content/drive/MyDrive/TRACKC_ULTRASOUND/"
            "TRACKC_FINAL_EXTERNAL_EVALUATION"
        )
    else:
        root = Path(
            "./TRACKC_FINAL_EXTERNAL_EVALUATION"
        )

    root.mkdir(parents=True, exist_ok=True)
    return root

DATA_ROOT = discover_data_root()
OUTPUT_ROOT = discover_output_root()
MANIFEST_PATH = DATA_ROOT / "roi_us_manifest.csv"

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "dataset": str,
        "split": str,
        "patient_id": str,
        "case_id": str,
        "sample_id": str,
        "global_patient_id": str,
        "npz_path": str,
    },
)

manifest["dataset"] = (
    manifest["dataset"].astype(str).str.strip().str.lower()
)
manifest["split"] = (
    manifest["split"].astype(str).str.strip().str.lower()
)
manifest["patient_id"] = (
    manifest["patient_id"].astype(str).str.strip()
)
manifest["sample_id"] = (
    manifest["sample_id"].astype(str).str.strip()
)

if "global_patient_id" not in manifest.columns:
    manifest["global_patient_id"] = (
        manifest["dataset"] + "::" + manifest["patient_id"]
    )

external = manifest[
    (manifest["dataset"] == "breast")
    & (manifest["split"] == "external")
].copy()

non_external_breast = manifest[
    (manifest["dataset"] == "breast")
    & (manifest["split"] != "external")
]
if len(non_external_breast):
    raise RuntimeError(
        "BrEaST contains rows outside the external split."
    )

if len(external) != EXPECTED_EXTERNAL_CROPS:
    raise RuntimeError(
        f"Expected {EXPECTED_EXTERNAL_CROPS} BrEaST external crops, "
        f"found {len(external)}."
    )

all_npz = list(DATA_ROOT.rglob("*.npz"))
by_name = defaultdict(list)
by_suffix = defaultdict(list)

for path in all_npz:
    by_name[path.name].append(path)
    normalized = str(path).replace("\\", "/")
    if "/npz/" in normalized:
        suffix = "npz/" + normalized.split("/npz/", 1)[1]
        by_suffix[suffix].append(path)

def unique_existing(
    paths: Iterable[Path],
    label: str,
) -> Optional[Path]:
    existing = []
    seen = set()

    for path in paths:
        path = Path(path)
        if not path.exists():
            continue
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            existing.append(path)

    if len(existing) == 1:
        return existing[0]
    if len(existing) > 1:
        raise RuntimeError(
            f"Ambiguous NPZ path resolution ({label}): {existing[:10]}"
        )
    return None

def resolve_external_npz(row: pd.Series) -> Path:
    stored = str(row["npz_path"]).replace("\\", "/")
    stored_path = Path(stored)
    sample_name = (
        str(row["sample_id"])
        if str(row["sample_id"]).endswith(".npz")
        else f"{row['sample_id']}.npz"
    )

    candidate = unique_existing(
        [stored_path],
        "stored_path",
    )
    if candidate is not None:
        return candidate

    deterministic = [
        DATA_ROOT
        / "npz"
        / "breast"
        / "external"
        / sample_name,
        DATA_ROOT
        / "npz"
        / "breast"
        / "external"
        / stored_path.name,
    ]
    candidate = unique_existing(
        deterministic,
        "deterministic",
    )
    if candidate is not None:
        return candidate

    suffixes = []
    if "/npz/" in stored:
        suffix = "npz/" + stored.split("/npz/", 1)[1]
        suffixes.extend(by_suffix.get(suffix, []))

    expected_suffix = (
        f"npz/breast/external/{sample_name}"
    )
    suffixes.extend(
        by_suffix.get(expected_suffix, [])
    )

    candidate = unique_existing(
        suffixes,
        "relative_suffix",
    )
    if candidate is not None:
        return candidate

    filename_matches = (
        by_name.get(sample_name, [])
        + by_name.get(stored_path.name, [])
    )
    candidate = unique_existing(
        filename_matches,
        "filename",
    )
    if candidate is not None:
        return candidate

    raise FileNotFoundError(
        f"Could not resolve external NPZ for sample_id={row['sample_id']}"
    )

external["resolved_npz_path"] = [
    str(resolve_external_npz(row))
    for _, row in tqdm(
        external.iterrows(),
        total=len(external),
        desc="Resolving BrEaST NPZ paths",
    )
]

if external["resolved_npz_path"].duplicated().any():
    raise RuntimeError(
        "Duplicate external NPZ paths detected."
    )

print(
    "External manifest audit:",
    {
        "crops": len(external),
        "patients": external["patient_id"].nunique(),
        "duplicates": int(
            external["resolved_npz_path"].duplicated().sum()
        ),
    },
)

In [ ]:
CHECKPOINT_EXTRACT_ROOT = Path(
    "/kaggle/working/TRACKC_CHECKPOINTS_EXTRACTED"
    if IN_KAGGLE
    else "/content/TRACKC_CHECKPOINTS_EXTRACTED"
)

EXPECTED_PARAMS = {
    "unet": 7_763_041,
    "attention_unet": 7_851_773,
    "swin_tiny_unet": 38_350_819,
}

CHECKPOINT_FILENAME_ALIASES = {
    "unet": [
        "unet",
    ],
    "attention_unet": [
        "attention_unet",
        "attentionunet",
        "att_unet",
        "attunet",
    ],
    "swin_tiny_unet": [
        "swin_tiny_unet",
        "swintinyunet",
        "swin_unet",
        "swinunet",
    ],
}

def zip_contains_checkpoint(
    zip_path: Path,
) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = [
                name.lower()
                for name in archive.namelist()
            ]

        return any(
            name.endswith("best.pt")
            for name in names
        )

    except Exception:
        return False

def extract_checkpoint_archives() -> None:
    CHECKPOINT_EXTRACT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    for base in search_roots():
        try:
            zip_paths = list(
                base.rglob("*.zip")
            )
        except Exception:
            zip_paths = []

        for zip_path in zip_paths:
            if not zip_contains_checkpoint(
                zip_path
            ):
                continue

            destination = (
                CHECKPOINT_EXTRACT_ROOT
                / re.sub(
                    r"[^A-Za-z0-9_.-]+",
                    "_",
                    zip_path.stem,
                )
            )

            try:
                safe_extract_zip(
                    zip_path,
                    destination,
                )
            except Exception as error:
                print(
                    "Checkpoint archive extraction warning:",
                    zip_path,
                    error,
                )

def checkpoint_candidates() -> List[Path]:
    roots = search_roots() + [
        CHECKPOINT_EXTRACT_ROOT
    ]

    if CHECKPOINT_ROOT_OVERRIDE:
        roots.insert(
            0,
            Path(
                CHECKPOINT_ROOT_OVERRIDE
            ),
        )

    candidates = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        try:
            paths = list(
                root.rglob("*.pt")
            )
        except Exception:
            paths = []

        for path in paths:
            try:
                key = str(
                    path.resolve()
                )
            except Exception:
                key = str(path)

            if key not in seen:
                seen.add(key)
                candidates.append(path)

    return candidates

def normalize_filename_token(
    value: str,
) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).lower(),
    ).strip("_")

def exact_filename_matches(
    path: Path,
    model_name: str,
    seed: int,
) -> bool:
    stem = normalize_filename_token(
        path.stem
    )

    aliases = CHECKPOINT_FILENAME_ALIASES[
        model_name
    ]

    accepted_stems = set()

    for alias in aliases:
        alias_normalized = (
            normalize_filename_token(
                alias
            )
        )

        accepted_stems.update(
            {
                f"{alias_normalized}_seed{seed}_best",
                f"{alias_normalized}_seed_{seed}_best",
                f"{alias_normalized}_{seed}_best",
                f"{alias_normalized}_best_seed{seed}",
                f"{alias_normalized}_best_seed_{seed}",
            }
        )

    return stem in accepted_stems

def read_checkpoint_metadata(
    path: Path,
) -> Dict:
    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except Exception as error:
        return {
            "load_ok": False,
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            )[:500],
        }

    metadata = {
        "load_ok": True,
        "model_name": None,
        "seed": None,
        "parameter_count": None,
        "epoch": None,
    }

    if isinstance(checkpoint, dict):
        metadata[
            "model_name"
        ] = checkpoint.get(
            "model_name"
        )
        metadata[
            "seed"
        ] = checkpoint.get(
            "seed"
        )
        metadata[
            "parameter_count"
        ] = checkpoint.get(
            "parameter_count"
        )
        metadata[
            "epoch"
        ] = checkpoint.get(
            "epoch"
        )

    return metadata

def metadata_matches_run(
    metadata: Dict,
    model_name: str,
    seed: int,
) -> bool:
    if not metadata.get(
        "load_ok",
        False,
    ):
        return False

    checkpoint_model = metadata.get(
        "model_name"
    )
    checkpoint_seed = metadata.get(
        "seed"
    )
    checkpoint_params = metadata.get(
        "parameter_count"
    )

    if checkpoint_model is not None:
        if str(checkpoint_model) != model_name:
            return False

    if checkpoint_seed is not None:
        try:
            if int(checkpoint_seed) != int(seed):
                return False
        except Exception:
            return False

    if checkpoint_params is not None:
        try:
            if int(checkpoint_params) != int(
                EXPECTED_PARAMS[model_name]
            ):
                return False
        except Exception:
            return False

    return True

def file_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    import hashlib

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

def select_checkpoint(
    all_paths: List[Path],
    model_name: str,
    seed: int,
) -> Tuple[Path, Dict]:
    exact_matches = [
        path
        for path in all_paths
        if exact_filename_matches(
            path,
            model_name,
            seed,
        )
    ]

    verified = []

    for path in exact_matches:
        metadata = read_checkpoint_metadata(
            path
        )

        if metadata_matches_run(
            metadata,
            model_name,
            seed,
        ):
            verified.append(
                (
                    path,
                    metadata,
                )
            )

    if not verified:
        for path in all_paths:
            if "best" not in path.stem.lower():
                continue

            metadata = read_checkpoint_metadata(
                path
            )

            if metadata_matches_run(
                metadata,
                model_name,
                seed,
            ):
                verified.append(
                    (
                        path,
                        metadata,
                    )
                )

    if not verified:
        raise FileNotFoundError(
            "No valid best checkpoint was found for "
            f"{model_name}, seed {seed}. "
            "Expected a filename such as "
            f"{model_name}_seed{seed}_best.pt."
        )

    if len(verified) == 1:
        return verified[0]

    hash_groups = defaultdict(list)

    for path, metadata in verified:
        hash_groups[
            file_sha256(path)
        ].append(
            (
                path,
                metadata,
            )
        )

    if len(hash_groups) == 1:
        identical_copies = next(
            iter(
                hash_groups.values()
            )
        )

        selected = sorted(
            identical_copies,
            key=lambda item: (
                len(str(item[0])),
                str(item[0]),
            ),
        )[0]

        print(
            f"Identical checkpoint copies found for "
            f"{model_name}, seed {seed}; "
            f"using: {selected[0]}"
        )

        return selected

    conflicting_paths = [
        str(path)
        for path, metadata in verified
    ]

    raise RuntimeError(
        "Multiple non-identical best checkpoints were found for "
        f"{model_name}, seed {seed}: "
        f"{conflicting_paths}. "
        "Keep only the intended checkpoint or set "
        "CHECKPOINT_ROOT_OVERRIDE."
    )

extract_checkpoint_archives()
all_checkpoint_paths = checkpoint_candidates()

print(
    "PT files discovered:",
    len(all_checkpoint_paths),
)

checkpoint_map = {}
checkpoint_metadata_map = {}

for run in FROZEN_RUNS:
    model_name = run[
        "model_name"
    ]
    seed = int(
        run["seed"]
    )

    checkpoint_path, metadata = select_checkpoint(
        all_checkpoint_paths,
        model_name,
        seed,
    )

    checkpoint_map[
        (
            model_name,
            seed,
        )
    ] = checkpoint_path

    checkpoint_metadata_map[
        (
            model_name,
            seed,
        )
    ] = metadata

checkpoint_rows = []

for (
    model_name,
    seed,
), path in checkpoint_map.items():
    metadata = checkpoint_metadata_map[
        (
            model_name,
            seed,
        )
    ]

    checkpoint_rows.append(
        {
            "model_name": model_name,
            "seed": seed,
            "checkpoint_path": str(
                path
            ),
            "size_mb": (
                path.stat().st_size
                / (1024 ** 2)
            ),
            "metadata_model_name": metadata.get(
                "model_name"
            ),
            "metadata_seed": metadata.get(
                "seed"
            ),
            "metadata_parameter_count": metadata.get(
                "parameter_count"
            ),
            "metadata_epoch": metadata.get(
                "epoch"
            ),
        }
    )

checkpoint_table = pd.DataFrame(
    checkpoint_rows
).sort_values(
    [
        "model_name",
        "seed",
    ]
)

if len(checkpoint_table) != 9:
    raise RuntimeError(
        f"Expected 9 checkpoints, found {len(checkpoint_table)}."
    )

if checkpoint_table[
    [
        "model_name",
        "seed",
    ]
].duplicated().any():
    raise RuntimeError(
        "Duplicate model/seed checkpoint assignments detected."
    )

print(
    "Checkpoint discovery and metadata validation: PASS"
)
display(
    checkpoint_table
)

In [ ]:
try:
    from torchvision.models import swin_t
    TORCHVISION_OK = True
except Exception as error:
    swin_t = None
    TORCHVISION_OK = False
    print("torchvision Swin import error:", error)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = ConvBlock(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.d4 = ConvBlock(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        center = self.center(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(center), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        return x * self.psi(psi)

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = ConvBlock(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.a4 = AttentionGate(base * 8, base * 8, base * 4)
        self.d4 = ConvBlock(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.a3 = AttentionGate(base * 4, base * 4, base * 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.a2 = AttentionGate(base * 2, base * 2, base)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.a1 = AttentionGate(base, base, base // 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        center = self.center(self.pool(e4))
        u4 = self.u4(center)
        d4 = self.d4(torch.cat([u4, self.a4(u4, e4)], 1))
        u3 = self.u3(d4)
        d3 = self.d3(torch.cat([u3, self.a3(u3, e3)], 1))
        u2 = self.u2(d3)
        d2 = self.d2(torch.cat([u2, self.a2(u2, e2)], 1))
        u1 = self.u1(d2)
        d1 = self.d1(torch.cat([u1, self.a1(u1, e1)], 1))
        return self.out(d1)

class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        if not TORCHVISION_OK or swin_t is None:
            raise RuntimeError(
                "torchvision.models.swin_t is unavailable"
            )
        self.swin = swin_t(weights=None)
        self.features = self.swin.features
        self.center = ConvBlock(768, 512)
        self.up3 = nn.ConvTranspose2d(512, 384, 2, 2)
        self.dec3 = ConvBlock(384 + 384, 256)
        self.up2 = nn.ConvTranspose2d(256, 192, 2, 2)
        self.dec2 = ConvBlock(192 + 192, 128)
        self.up1 = nn.ConvTranspose2d(128, 96, 2, 2)
        self.dec1 = ConvBlock(96 + 96, 64)
        self.up0 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec0 = ConvBlock(32, 32)
        self.up_final = nn.ConvTranspose2d(32, 32, 2, 2)
        self.out = nn.Conv2d(32, out_ch, 1)

    def _to_nchw(self, x):
        if x.ndim == 4 and x.shape[1] not in [96, 192, 384, 768]:
            return x.permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        feats = []
        y = x
        for index, layer in enumerate(self.features):
            y = layer(y)
            if index in [1, 3, 5, 7]:
                feats.append(self._to_nchw(y))

        f1, f2, f3, f4 = feats
        center = self.center(f4)
        d3 = self.dec3(torch.cat([self.up3(center), f3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), f2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), f1], 1))
        d0 = self.dec0(self.up0(d1))
        output = self.out(self.up_final(d0))

        if output.shape[-2:] != x.shape[-2:]:
            output = F.interpolate(
                output,
                size=x.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return output

def build_model(model_name: str) -> nn.Module:
    if model_name == "unet":
        return UNet(in_ch=3, out_ch=1)
    if model_name == "attention_unet":
        return AttentionUNet(in_ch=3, out_ch=1)
    if model_name == "swin_tiny_unet":
        return SwinTinyUNet(out_ch=1)
    raise ValueError(model_name)

EXPECTED_PARAMS = {
    "unet": 7_763_041,
    "attention_unet": 7_851_773,
    "swin_tiny_unet": 38_350_819,
}

In [ ]:
def extract_state_dict(checkpoint_object):
    if isinstance(checkpoint_object, dict):
        for key in [
            "model_state",
            "model_state_dict",
            "state_dict",
            "best_model_state_dict",
        ]:
            value = checkpoint_object.get(key)
            if isinstance(value, dict) and value:
                return value

        if checkpoint_object and all(
            torch.is_tensor(value)
            for value in checkpoint_object.values()
        ):
            return checkpoint_object

    raise RuntimeError(
        "No model state dictionary was found in the checkpoint."
    )

def normalize_state_dict_keys(state_dict):
    prefixes = [
        "module.",
        "_orig_mod.",
        "model.",
        "net.",
    ]

    variants = [state_dict]

    for prefix in prefixes:
        if all(key.startswith(prefix) for key in state_dict):
            variants.append(
                {
                    key[len(prefix):]: value
                    for key, value in state_dict.items()
                }
            )

    return variants

def load_exact_model(
    model_name: str,
    seed: int,
) -> nn.Module:
    model = build_model(model_name)

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )
    if parameter_count != EXPECTED_PARAMS[model_name]:
        raise RuntimeError(
            f"Parameter count mismatch for {model_name}: "
            f"{parameter_count} != {EXPECTED_PARAMS[model_name]}"
        )

    checkpoint_path = checkpoint_map[
        (model_name, seed)
    ]
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    state_dict = extract_state_dict(checkpoint)

    model_keys = set(model.state_dict().keys())
    loaded = False
    errors = []

    for variant in normalize_state_dict_keys(state_dict):
        if set(variant.keys()) != model_keys:
            continue

        shape_mismatch = [
            key
            for key in model_keys
            if tuple(variant[key].shape)
            != tuple(model.state_dict()[key].shape)
        ]
        if shape_mismatch:
            errors.append(
                f"shape_mismatch={shape_mismatch[:10]}"
            )
            continue

        model.load_state_dict(
            variant,
            strict=True,
        )
        loaded = True
        break

    if not loaded:
        raise RuntimeError(
            f"Strict checkpoint loading failed for {model_name}, seed {seed}. "
            f"Errors={errors[:5]}"
        )

    model = model.to(DEVICE)
    model.eval()

    print(
        f"Loaded {model_name}, seed {seed}, "
        f"threshold={float(frozen_df.loc[(frozen_df.model_name==model_name)&(frozen_df.seed==seed),'selected_threshold'].iloc[0]):.2f}, "
        f"parameters={parameter_count:,}"
    )
    return model

In [ ]:
class ExternalUltrasoundDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        with np.load(row["resolved_npz_path"]) as data:
            image = data["image"].astype(np.float32)
            mask = data["mask"].astype(np.uint8)

        image_tensor = torch.from_numpy(
            np.stack([image, image, image], axis=0)
        ).float()
        mask_tensor = torch.from_numpy(
            mask[None, ...]
        ).float()

        metadata = {
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "global_patient_id": str(row["global_patient_id"]),
            "case_id": str(row["case_id"]),
            "pathology": str(row.get("pathology", "")),
            "diagnosis": str(row.get("diagnosis", "")),
            "birads": str(row.get("birads", "")),
            "crop_touches_border": int(
                row.get("crop_touches_border", 0)
            ),
            "lesion_ratio_crop": float(
                row.get("lesion_ratio_crop", np.nan)
            ),
            "crop_size_native": float(
                row.get("crop_size_native", np.nan)
            ),
            "npz_path": str(row["resolved_npz_path"]),
        }

        return image_tensor, mask_tensor, metadata

external_dataset = ExternalUltrasoundDataset(external)

In [ ]:
def binary_confusion(prediction, target):
    prediction = prediction.astype(bool)
    target = target.astype(bool)
    tp = int(np.logical_and(prediction, target).sum())
    fp = int(np.logical_and(prediction, ~target).sum())
    fn = int(np.logical_and(~prediction, target).sum())
    tn = int(np.logical_and(~prediction, ~target).sum())
    return tp, fp, fn, tn

def metrics_from_counts(tp, fp, fn, tn, smooth=1e-8):
    return {
        "dice": (2 * tp + smooth)
        / (2 * tp + fp + fn + smooth),
        "iou": (tp + smooth)
        / (tp + fp + fn + smooth),
        "precision": (tp + smooth)
        / (tp + fp + smooth),
        "recall": (tp + smooth)
        / (tp + fn + smooth),
    }

def surface_distances(prediction, target):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    if not prediction.any() or not target.any():
        return np.array([], dtype=np.float32)

    prediction_surface = np.logical_xor(
        prediction,
        binary_erosion(prediction),
    )
    target_surface = np.logical_xor(
        target,
        binary_erosion(target),
    )

    distance_to_target = distance_transform_edt(
        ~target_surface
    )
    distance_to_prediction = distance_transform_edt(
        ~prediction_surface
    )

    return np.concatenate([
        distance_to_target[prediction_surface],
        distance_to_prediction[target_surface],
    ]).astype(np.float32)

def hd95_asd(prediction, target):
    distances = surface_distances(prediction, target)
    if distances.size == 0:
        return np.nan, np.nan
    return (
        float(np.percentile(distances, 95)),
        float(np.mean(distances)),
    )

def metadata_item(metadata, key, index):
    value = metadata[key]
    if torch.is_tensor(value):
        return value[index].item()
    if isinstance(value, (list, tuple)):
        return value[index]
    return value

In [ ]:
all_crop_frames = []
all_patient_frames = []
all_run_summaries = []
qualitative_cache = {}

for run in FROZEN_RUNS:
    model_name = run["model_name"]
    seed = int(run["seed"])
    threshold = float(run["selected_threshold"])

    print(
        "\nEvaluating:",
        {
            "model_name": model_name,
            "seed": seed,
            "threshold": threshold,
        },
    )

    model = load_exact_model(
        model_name,
        seed,
    )

    loader = DataLoader(
        external_dataset,
        batch_size=BATCH_SIZE_BY_MODEL[model_name],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=AMP_ENABLED,
        persistent_workers=NUM_WORKERS > 0,
    )

    crop_rows = []
    cached_examples = []

    with torch.inference_mode():
        for images, masks, metadata in tqdm(
            loader,
            desc=f"{model_name} seed {seed}",
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

            if AMP_ENABLED:
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    logits = model(images)
            else:
                logits = model(images)

            probabilities = (
                torch.sigmoid(logits)
                .detach()
                .cpu()
                .numpy()[:, 0]
            )
            targets = (
                masks.numpy()[:, 0]
                .astype(np.uint8)
            )

            for index in range(len(probabilities)):
                probability = probabilities[index]
                target = targets[index]
                prediction = (
                    probability >= threshold
                ).astype(np.uint8)

                tp, fp, fn, tn = binary_confusion(
                    prediction,
                    target,
                )
                metrics = metrics_from_counts(
                    tp, fp, fn, tn
                )
                hd95, asd = hd95_asd(
                    prediction,
                    target,
                )

                row = {
                    "model_name": model_name,
                    "seed": seed,
                    "threshold": threshold,
                    "sample_id": str(
                        metadata_item(
                            metadata,
                            "sample_id",
                            index,
                        )
                    ),
                    "patient_id": str(
                        metadata_item(
                            metadata,
                            "patient_id",
                            index,
                        )
                    ),
                    "global_patient_id": str(
                        metadata_item(
                            metadata,
                            "global_patient_id",
                            index,
                        )
                    ),
                    "case_id": str(
                        metadata_item(
                            metadata,
                            "case_id",
                            index,
                        )
                    ),
                    "pathology": str(
                        metadata_item(
                            metadata,
                            "pathology",
                            index,
                        )
                    ),
                    "diagnosis": str(
                        metadata_item(
                            metadata,
                            "diagnosis",
                            index,
                        )
                    ),
                    "birads": str(
                        metadata_item(
                            metadata,
                            "birads",
                            index,
                        )
                    ),
                    "crop_touches_border": int(
                        metadata_item(
                            metadata,
                            "crop_touches_border",
                            index,
                        )
                    ),
                    "lesion_ratio_crop": float(
                        metadata_item(
                            metadata,
                            "lesion_ratio_crop",
                            index,
                        )
                    ),
                    "crop_size_native": float(
                        metadata_item(
                            metadata,
                            "crop_size_native",
                            index,
                        )
                    ),
                    "tp": tp,
                    "fp": fp,
                    "fn": fn,
                    "tn": tn,
                    **metrics,
                    "hd95_px": hd95,
                    "asd_px": asd,
                    "prediction_empty": int(
                        prediction.sum() == 0
                    ),
                    "target_pixels": int(
                        target.sum()
                    ),
                    "prediction_pixels": int(
                        prediction.sum()
                    ),
                }
                crop_rows.append(row)

                if (
                    CREATE_QUALITATIVE_OVERLAYS
                    and seed == 42
                    and len(cached_examples)
                    < N_OVERLAYS_PER_MODEL
                ):
                    cached_examples.append({
                        "sample_id": row["sample_id"],
                        "image": images[index, 0]
                        .detach().cpu().numpy(),
                        "target": target,
                        "prediction": prediction,
                        "dice": row["dice"],
                    })

    crop_frame = pd.DataFrame(crop_rows)

    patient_rows = []
    for patient_key, group in crop_frame.groupby(
        "global_patient_id",
        sort=False,
    ):
        tp = int(group["tp"].sum())
        fp = int(group["fp"].sum())
        fn = int(group["fn"].sum())
        tn = int(group["tn"].sum())

        patient_rows.append({
            "model_name": model_name,
            "seed": seed,
            "threshold": threshold,
            "global_patient_id": patient_key,
            "patient_id": str(group["patient_id"].iloc[0]),
            "pathology": str(group["pathology"].iloc[0]),
            "diagnosis": str(group["diagnosis"].iloc[0]),
            "birads": str(group["birads"].iloc[0]),
            "n_crops": int(len(group)),
            "crop_touches_border_any": int(
                group["crop_touches_border"].max()
            ),
            "lesion_ratio_crop_mean": float(
                group["lesion_ratio_crop"].mean()
            ),
            "crop_size_native_mean": float(
                group["crop_size_native"].mean()
            ),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            **metrics_from_counts(tp, fp, fn, tn),
        })

    patient_frame = pd.DataFrame(patient_rows)

    run_summary = {
        "model_name": model_name,
        "seed": seed,
        "threshold": threshold,
        "n_crops": int(len(crop_frame)),
        "n_patients": int(len(patient_frame)),
        "crop_dice_mean": float(crop_frame["dice"].mean()),
        "crop_iou_mean": float(crop_frame["iou"].mean()),
        "crop_precision_mean": float(
            crop_frame["precision"].mean()
        ),
        "crop_recall_mean": float(
            crop_frame["recall"].mean()
        ),
        "crop_hd95_mean_px": float(
            crop_frame["hd95_px"].mean()
        ),
        "crop_asd_mean_px": float(
            crop_frame["asd_px"].mean()
        ),
        "patient_dice_mean": float(
            patient_frame["dice"].mean()
        ),
        "patient_iou_mean": float(
            patient_frame["iou"].mean()
        ),
        "patient_precision_mean": float(
            patient_frame["precision"].mean()
        ),
        "patient_recall_mean": float(
            patient_frame["recall"].mean()
        ),
        "prediction_empty_count": int(
            crop_frame["prediction_empty"].sum()
        ),
    }

    run_dir = (
        OUTPUT_ROOT
        / f"{model_name}_seed{seed}"
    )
    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    crop_frame.to_csv(
        run_dir / "external_crop_metrics.csv",
        index=False,
    )
    patient_frame.to_csv(
        run_dir / "external_patient_metrics.csv",
        index=False,
    )
    (run_dir / "external_run_summary.json").write_text(
        json.dumps(run_summary, indent=2),
        encoding="utf-8",
    )

    all_crop_frames.append(crop_frame)
    all_patient_frames.append(patient_frame)
    all_run_summaries.append(run_summary)

    if cached_examples:
        qualitative_cache[model_name] = cached_examples

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

all_crop = pd.concat(
    all_crop_frames,
    ignore_index=True,
)
all_patient = pd.concat(
    all_patient_frames,
    ignore_index=True,
)
run_summary_df = pd.DataFrame(
    all_run_summaries
).sort_values(
    ["model_name", "seed"]
)

all_crop.to_csv(
    OUTPUT_ROOT / "all_external_crop_metrics.csv",
    index=False,
)
all_patient.to_csv(
    OUTPUT_ROOT / "all_external_patient_metrics.csv",
    index=False,
)
run_summary_df.to_csv(
    OUTPUT_ROOT / "external_model_seed_summary.csv",
    index=False,
)

display(run_summary_df)

In [ ]:
summary_metrics = [
    "crop_dice_mean",
    "crop_iou_mean",
    "crop_precision_mean",
    "crop_recall_mean",
    "crop_hd95_mean_px",
    "crop_asd_mean_px",
    "patient_dice_mean",
    "patient_iou_mean",
    "patient_precision_mean",
    "patient_recall_mean",
]

aggregate_rows = []

for model_name, group in run_summary_df.groupby(
    "model_name",
    sort=False,
):
    row = {
        "model_name": model_name,
        "n_seeds": int(len(group)),
    }

    for metric in summary_metrics:
        row[f"{metric}_mean"] = float(
            group[metric].mean()
        )
        row[f"{metric}_sd"] = float(
            group[metric].std(ddof=1)
        )

    row["prediction_empty_count_total"] = int(
        group["prediction_empty_count"].sum()
    )
    aggregate_rows.append(row)

aggregate_summary = pd.DataFrame(
    aggregate_rows
).sort_values(
    "patient_dice_mean_mean",
    ascending=False,
)

aggregate_summary.to_csv(
    OUTPUT_ROOT / "external_model_mean_sd_summary.csv",
    index=False,
)

display(aggregate_summary)

In [ ]:
patient_metric_columns = [
    "dice",
    "iou",
    "precision",
    "recall",
]

patient_metadata_columns = [
    "global_patient_id",
    "patient_id",
    "pathology",
    "diagnosis",
    "birads",
    "crop_touches_border_any",
    "lesion_ratio_crop_mean",
    "crop_size_native_mean",
]

patient_metadata = (
    all_patient[
        patient_metadata_columns
    ]
    .drop_duplicates(
        "global_patient_id"
    )
)

seed_averaged = (
    all_patient
    .groupby(
        [
            "model_name",
            "global_patient_id",
        ],
        as_index=False,
    )[patient_metric_columns]
    .mean()
    .merge(
        patient_metadata,
        on="global_patient_id",
        how="left",
    )
)

seed_averaged.to_csv(
    OUTPUT_ROOT
    / "external_patient_metrics_seed_averaged.csv",
    index=False,
)

print(
    "Seed-averaged rows:",
    len(seed_averaged),
)

In [ ]:
MODEL_PAIRS = [
    ("swin_tiny_unet", "unet"),
    ("swin_tiny_unet", "attention_unet"),
    ("attention_unet", "unet"),
]

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

comparison_rows = []

for metric in [
    "dice",
    "iou",
    "precision",
    "recall",
]:
    wide = seed_averaged.pivot(
        index="global_patient_id",
        columns="model_name",
        values=metric,
    ).dropna()

    for model_a, model_b in MODEL_PAIRS:
        differences = (
            wide[model_a].to_numpy()
            - wide[model_b].to_numpy()
        )

        n = len(differences)
        bootstrap_means = np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        )

        for replicate in range(
            BOOTSTRAP_REPLICATES
        ):
            indices = rng.integers(
                0,
                n,
                size=n,
            )
            bootstrap_means[replicate] = (
                differences[indices].mean()
            )

        ci_low, ci_high = np.percentile(
            bootstrap_means,
            [2.5, 97.5],
        )

        try:
            statistic, p_value = wilcoxon(
                differences,
                zero_method="wilcox",
                alternative="two-sided",
                mode="auto",
            )
        except ValueError:
            statistic, p_value = np.nan, 1.0

        comparison_rows.append({
            "metric": metric,
            "model_a": model_a,
            "model_b": model_b,
            "n_patients": n,
            "mean_difference_a_minus_b": float(
                differences.mean()
            ),
            "median_difference_a_minus_b": float(
                np.median(differences)
            ),
            "bootstrap_ci95_low": float(ci_low),
            "bootstrap_ci95_high": float(ci_high),
            "wilcoxon_statistic": float(statistic),
            "wilcoxon_p_raw": float(p_value),
        })

comparisons = pd.DataFrame(
    comparison_rows
)

comparisons["wilcoxon_p_holm"] = np.nan
comparisons["holm_reject_0_05"] = False

for metric, indices in comparisons.groupby(
    "metric"
).groups.items():
    raw_p = comparisons.loc[
        indices,
        "wilcoxon_p_raw",
    ].to_numpy()

    reject, adjusted, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method="holm",
    )

    comparisons.loc[
        indices,
        "wilcoxon_p_holm",
    ] = adjusted
    comparisons.loc[
        indices,
        "holm_reject_0_05",
    ] = reject

comparisons.to_csv(
    OUTPUT_ROOT
    / "external_paired_model_comparisons.csv",
    index=False,
)

display(comparisons)

In [ ]:
model_ci_rows = []

for model_name, group in seed_averaged.groupby(
    "model_name"
):
    for metric in [
        "dice",
        "iou",
        "precision",
        "recall",
    ]:
        values = group[metric].to_numpy()
        n = len(values)

        bootstrap_means = np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        )

        for replicate in range(
            BOOTSTRAP_REPLICATES
        ):
            indices = rng.integers(
                0,
                n,
                size=n,
            )
            bootstrap_means[replicate] = (
                values[indices].mean()
            )

        ci_low, ci_high = np.percentile(
            bootstrap_means,
            [2.5, 97.5],
        )

        model_ci_rows.append({
            "model_name": model_name,
            "metric": metric,
            "n_patients": n,
            "mean": float(values.mean()),
            "sd_between_patients": float(
                values.std(ddof=1)
            ),
            "bootstrap_ci95_low": float(ci_low),
            "bootstrap_ci95_high": float(ci_high),
        })

model_confidence_intervals = pd.DataFrame(
    model_ci_rows
)

model_confidence_intervals.to_csv(
    OUTPUT_ROOT
    / "external_model_bootstrap_confidence_intervals.csv",
    index=False,
)

display(model_confidence_intervals)

In [ ]:
subgroup_rows = []

analysis_frame = seed_averaged.copy()

analysis_frame["lesion_size_quartile"] = pd.qcut(
    analysis_frame[
        "lesion_ratio_crop_mean"
    ],
    q=4,
    labels=[
        "Q1_smallest",
        "Q2",
        "Q3",
        "Q4_largest",
    ],
    duplicates="drop",
)

subgroup_definitions = {
    "pathology": "pathology",
    "border_touch": "crop_touches_border_any",
    "lesion_size_quartile": "lesion_size_quartile",
}

for subgroup_name, column in subgroup_definitions.items():
    for (
        model_name,
        subgroup_value,
    ), group in analysis_frame.groupby(
        ["model_name", column],
        dropna=False,
    ):
        subgroup_rows.append({
            "subgroup": subgroup_name,
            "subgroup_value": str(subgroup_value),
            "model_name": model_name,
            "n_patients": int(len(group)),
            "dice_mean": float(group["dice"].mean()),
            "dice_sd": float(group["dice"].std(ddof=1)),
            "iou_mean": float(group["iou"].mean()),
            "precision_mean": float(
                group["precision"].mean()
            ),
            "recall_mean": float(
                group["recall"].mean()
            ),
        })

subgroup_summary = pd.DataFrame(
    subgroup_rows
)

subgroup_summary.to_csv(
    OUTPUT_ROOT
    / "external_subgroup_summary.csv",
    index=False,
)

display(subgroup_summary)

In [ ]:
if CREATE_QUALITATIVE_OVERLAYS:
    import matplotlib.pyplot as plt

    qualitative_root = (
        OUTPUT_ROOT / "qualitative_overlays"
    )
    qualitative_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for model_name, examples in qualitative_cache.items():
        rows = len(examples)
        figure, axes = plt.subplots(
            rows,
            4,
            figsize=(14, 3 * rows),
        )

        if rows == 1:
            axes = np.array([axes])

        for index, example in enumerate(examples):
            image = example["image"]
            target = example["target"]
            prediction = example["prediction"]

            overlay = np.stack(
                [image, image, image],
                axis=-1,
            )
            overlay[..., 0] = np.maximum(
                overlay[..., 0],
                target * 0.9,
            )
            overlay[..., 1] = np.maximum(
                overlay[..., 1],
                prediction * 0.7,
            )

            axes[index, 0].imshow(
                image,
                cmap="gray",
            )
            axes[index, 0].set_title(
                example["sample_id"]
            )
            axes[index, 1].imshow(
                target,
                cmap="gray",
            )
            axes[index, 1].set_title(
                "Ground truth"
            )
            axes[index, 2].imshow(
                prediction,
                cmap="gray",
            )
            axes[index, 2].set_title(
                f"Prediction | Dice={example['dice']:.3f}"
            )
            axes[index, 3].imshow(overlay)
            axes[index, 3].set_title(
                "Red=GT, Green=prediction"
            )

            for axis in axes[index]:
                axis.axis("off")

        figure.tight_layout()
        figure_path = (
            qualitative_root
            / f"{model_name}_seed42_overlays.png"
        )
        figure.savefig(
            figure_path,
            dpi=160,
            bbox_inches="tight",
        )
        plt.close(figure)

    print(
        "Qualitative overlays:",
        qualitative_root,
    )

In [ ]:
winner_row = aggregate_summary.iloc[0]
winner_model = str(
    winner_row["model_name"]
)

final_summary = {
    "status": "completed_sealed_external_evaluation",
    "external_dataset": "BrEaST",
    "n_external_crops": int(len(external)),
    "n_external_patients": int(
        external["patient_id"].nunique()
    ),
    "runs_evaluated": int(len(run_summary_df)),
    "threshold_source": "BUS-BRA validation, patient-level",
    "external_threshold_search_performed": False,
    "winner_by_mean_patient_dice": winner_model,
    "aggregate_results": aggregate_summary.to_dict(
        orient="records"
    ),
}

(OUTPUT_ROOT / "FINAL_EXTERNAL_SUMMARY.json").write_text(
    json.dumps(final_summary, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# Track C — Final sealed BrEaST external evaluation",
    "",
    f"- Status: {final_summary['status']}",
    f"- External crops: {final_summary['n_external_crops']}",
    f"- External patients: {final_summary['n_external_patients']}",
    f"- Runs: {final_summary['runs_evaluated']}",
    "- Threshold source: BUS-BRA validation at patient level",
    "- External threshold search: NO",
    f"- Highest mean external patient Dice: {winner_model}",
    "",
    "## Mean ± SD across seeds",
    "",
    aggregate_summary.to_markdown(index=False),
]

(OUTPUT_ROOT / "FINAL_EXTERNAL_REPORT.md").write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

final_zip = Path(
    "/kaggle/working/TRACKC_FINAL_EXTERNAL_EVALUATION.zip"
    if IN_KAGGLE
    else str(OUTPUT_ROOT.parent / "TRACKC_FINAL_EXTERNAL_EVALUATION.zip")
)

with zipfile.ZipFile(
    final_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    allowZip64=True,
) as archive:
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(OUTPUT_ROOT.parent),
            )

print(
    json.dumps(
        final_summary,
        indent=2,
    )
)
print(
    "\nDownloadable final ZIP:",
    final_zip,
)